author: baptisteD

# Sprint 1 — Ingestion : carte d'identité des fichiers sources

## Objectif

Avant d'exploiter les données, je veux savoir **ce que contient chaque fichier** : sa taille, son encodage, son nombre de lignes et de colonnes, le type de chaque colonne et quelques statistiques (min, max, moyenne, médiane).

Les sources sont dans `Datas/` :
- `releves_incidents.csv` : les relevés d'incidents saisis par les opérateurs,
- `telemetry.csv` : la télémétrie horaire des machines,
- `machine.sql` : un **script** PostgreSQL (référentiel machines + maintenance), donc pas un fichier tabulaire.

## Démarche générale

1. Plutôt que d'écrire le traitement directement dans le notebook, je l'ai mis dans des **classes Python** du package `src/indusense/`. Je peux ainsi les réutiliser sur n'importe quel fichier, les tester et les appeler depuis un script. Le notebook sert uniquement à les piloter et à expliquer la démarche.
2. Je traite d'abord les **CSV** avec `IdentityGenerator`, qui analyse le fichier, et `IdentityMarkdown`, qui met en forme la carte.
3. Pour le **SQL**, je ne réécris pas un second analyseur : `DumpToCsv` extrait chaque table du dump dans un DataFrame, que je donne **directement** à `IdentityGenerator`, sans repasser par un fichier CSV. Toutes les sources produisent ainsi des cartes au même format, et la carte décrit bien le dump d'origine.

### Import des classes

Le package est dans `src/`, je l'ajoute donc au `sys.path` pour pouvoir l'importer depuis le notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))

from indusense.identity_generator import IdentityGenerator
from indusense.identity_markdown import ColumnStats, IdentityMarkdown
from indusense.dump_to_csv import DumpToCsv

## 1. La carte d'identité d'un CSV : `IdentityGenerator`

### Où sont écrits les fichiers ?

`artifacts/ingestion/identities/<YYYYMMDD>/<name>_<HHMMSS>.md` (`name` = nom du fichier source sans extension, modifiable par setter)

- un **dossier par jour** pour regrouper les exécutions d'une même journée,
- l'**heure dans le nom du fichier** pour ne jamais écraser une carte précédente : je garde l'historique et je peux comparer deux passages,
- du **Markdown**, lisible directement dans VS Code ou sur un dépôt Git.

### Mes choix pour chaque information

| Information | Comment je l'obtiens | Pourquoi |
| --- | --- | --- |
| **Source** | Chemin **relatif à la racine du projet** (le dossier qui contient `pyproject.toml`) | Un chemin absolu (`C:\wamp\...`) ne vaut que sur ma machine ; le chemin relatif reste valable pour toute l'équipe. |
| **Filesize** | Taille en octets convertie en o / Ko / Mo / Go | Plus lisible qu'un nombre d'octets brut. |
| **ProcessBeginAt / EndAt** | Heure avant la lecture / après l'analyse | Mesure le temps de traitement de chaque fichier. |
| **Charset** | J'essaie de décoder le fichier en `utf-8`, puis en `iso-8859-1` | UTF-8 est strict : s'il décode sans erreur, c'est le bon encodage. Sinon, c'est généralement un fichier Latin-1 (Excel sous Windows). Pas besoin de librairie supplémentaire. |
| **Rows / Cols** | `len(df)` et `len(df.columns)` après lecture par pandas | |
| **Séparateur** | Paramètre `separator` du constructeur (`,` par défaut) | Un CSV peut utiliser `,`, `;` ou la tabulation : je ne le code pas en dur. |

### Inférence du type des colonnes

Je teste les règles dans cet ordre :
1. nom qui se termine par `_id` → **IDENTITY** (un identifiant, même numérique, n'a pas de sens statistique),
2. type booléen pandas → **BOOLEAN**,
3. type numérique pandas → **NUMERIC**,
4. les 100 premières valeurs commencent par `AAAA-MM-JJ` → **DATE** (pandas lit les dates comme du texte, d'où ce test),
5. sinon → **TEXT**.

### Statistiques

- **NUMERIC** : min, max, moyenne, médiane,
- **DATE** : min et max uniquement (une moyenne de dates n'a pas d'intérêt ici),
- les autres types sont ignorés, comme le demande la consigne.

In [ ]:
generator = IdentityGenerator()  # séparateur "," par défaut

for csv in ["Datas/releves_incidents.csv", "Datas/telemetry.csv"]:
    print(generator.generateIdentities(csv))

### Séparer l'analyse de la mise en forme : `IdentityMarkdown`

Au départ, `IdentityGenerator` construisait lui-même les lignes du Markdown. Il faisait donc deux métiers : **analyser** le fichier et **mettre en forme** le résultat. J'ai sorti le modèle Markdown dans une classe à part, `IdentityMarkdown` (`src/indusense/identity_markdown.py`) :

- **`IdentityGenerator`** collecte les données brutes : taille en octets, `datetime` de début et de fin, encodage, nombre de lignes et de colonnes, types et statistiques ;
- **`IdentityMarkdown`** les reçoit en entrée et s'occupe uniquement de la présentation : unités de taille, format `HH:MM:SS`, tableaux, arrondi de la moyenne.

Ce que j'y gagne :
- changer le gabarit (ajouter une ligne, passer en HTML…) ne touche plus à l'analyse, et inversement ;
- la mise en forme se teste **sans lire de fichier**, en lui passant des valeurs à la main ;
- le même modèle pourra servir à d'autres sources, comme le SQL, du moment qu'on lui fournit les mêmes données.

J'ai choisi une **`dataclass`** : chaque donnée est un attribut nommé et typé, et elles sont toutes **optionnelles**. Les statistiques d'une colonne sont regroupées dans une petite `dataclass` `ColumnStats` (nom, min, max, moyenne, médiane, et un indicateur `is_date`).

### Donnée manquante → « Inconnu »

Si une donnée n'est pas fournie, le Markdown affiche **`Inconnu`** au lieu de planter ou d'écrire `None`. Je considère comme manquant :
- `None` ou une chaîne vide,
- `NaN` (une statistique sur une colonne vide) et `NaT` (une date qui n'a pas pu être lue) : pandas renvoie ces valeurs au lieu de `None`, il faut donc les tester aussi.

Deux cas particuliers :
- si l'encodage n'est pas détecté, le générateur renvoie maintenant `None` au lieu de `"unknown"`, ce qui affiche `Inconnu` comme pour les autres données ;
- pour une colonne **date**, la moyenne et la médiane affichent `-` et non `Inconnu`, car elles ne sont pas *inconnues* mais *non applicables*. `Inconnu` est réservé à une donnée qui aurait dû être là.

Exemple avec des données volontairement incomplètes :

In [ ]:
print(IdentityMarkdown(
    source="Datas/exemple.csv",
    rows=10,
    stats=[ColumnStats("temperature_c", min=32.0, max=float("nan"), median=48.0)],
).render())

### Séparer la récupération des données de la construction du Markdown

`generateIdentities` faisait tout d'un bloc : lire le fichier, calculer, construire le Markdown et l'écrire. Impossible donc de lui donner des données qui ne viennent pas d'un CSV, ou de corriger une valeur avant l'écriture. J'ai découpé le générateur en étapes, une méthode chacune :

| Étape | Méthode | Rôle |
| --- | --- | --- |
| 1. Récupération | `collectFromFile(source)` | lit un CSV et renseigne **tous** les attributs |
| | `collectFromDataFrame(df)` | renseigne ce qui se calcule depuis les données : lignes, colonnes, types, stats, début et fin |
| 2. Construction | `buildMarkdown()` | construit le Markdown à partir des attributs courants (via `IdentityMarkdown`) |
| Écriture | `write()` | écrit le Markdown dans `artifacts/…` |

`generateIdentities(source)` reste disponible comme raccourci (`collectFromFile` + `write`) : le code existant continue de fonctionner.

### Un getter et un setter par attribut

Entre les deux étapes, les données sont stockées dans des attributs du générateur : `name`, `source`, `filesize`, `begin`, `end`, `charset`, `rows`, `cols`, `columns`, `stats`. Chacun a un **getter et un setter**, que j'ai écrits sous forme de **`@property`**, la façon idiomatique en Python : on écrit `generator.charset = "utf-8"` et le setter est appelé.

Ce que ça permet :
- **lire** n'importe quelle donnée collectée avant de construire la carte ;
- **compléter** ce que la récupération ne peut pas savoir (la source d'un DataFrame, par exemple) ;
- **corriger** une valeur, comme le type d'une colonne mal inféré.

Le setter peut aussi normaliser la valeur : `source` accepte un `Path` et le stocke au format `a/b/c`, et `columns` et `stats` sont copiés pour qu'une modification extérieure n'altère pas le générateur.

`reset()` vide tous les attributs. Une donnée qu'on n'a pas renseignée s'affiche donc `Inconnu` dans la carte, conformément à la règle précédente.

In [ ]:
# Récupération, lecture par getter, correction par setter, puis construction
generator.collectFromFile("Datas/releves_incidents.csv")
print(generator.rows, generator.cols, generator.charset)

columns = generator.columns
columns["operator_badge"] = "IDENTITY"   # un badge est un identifiant, pas du texte
generator.columns = columns

print(generator.buildMarkdown()[:600])

## 2. Le cas du script SQL : `DumpToCsv`

### Le problème

`machine.sql` n'est pas un tableau : c'est un script qui contient des `BEGIN`, des `CREATE TABLE`, des `INSERT` et un `COMMIT`. Je ne peux pas le lire avec `pd.read_csv`, et le découper avec des expressions régulières serait fragile (virgules ou apostrophes dans les valeurs, `INSERT` sur plusieurs lignes…).

### La solution : un vrai parseur SQL

J'ai choisi **[sqlglot](https://github.com/tobymao/sqlglot)** (`uv add sqlglot`) :
- c'est du Python pur, sans base de données à installer,
- il connaît le **dialecte PostgreSQL** du dump (paramètre `dialect`),
- il transforme le script en **arbre syntaxique** (AST) : je manipule des objets `Insert`, `Values`, `Schema` au lieu de chaînes de caractères.

### Le raisonnement de `extractTables`

1. Je parse tout le script et je parcours les instructions une par une.
2. Je ne garde **que les `INSERT ... VALUES`**. `BEGIN`, `CREATE TABLE`, `COMMIT` ou `INSERT ... SELECT` ne contiennent pas de données à extraire.
3. Pour chaque `INSERT`, je récupère le **nom de la table** et, s'il y en a une, la **liste des colonnes**.
4. Chaque ligne de `VALUES` devient une ligne de DataFrame. Les littéraux SQL sont convertis en valeurs Python (`'abc'` → `str`, `12` → `int`, `TRUE` → `bool`, `NULL` → `None`). Un nombre décimal (`2.64`) arrive en `Decimal`, que pandas range dans une colonne `object` : la colonne serait alors classée TEXT. Je le convertis donc en `float`. Pour une expression non littérale comme `NOW()`, je garde le texte SQL plutôt que de planter.
5. Une même table peut avoir **plusieurs `INSERT`** : je les **concatène** pour obtenir un seul DataFrame par table.

### Écriture (optionnelle)

`convert` écrit **un CSV par table** dans `Datas/dump/<table>_<YYYYMMDD_HHMMSS>.csv`. Le séparateur et l'encodage sont paramétrables, comme pour `IdentityGenerator`. Cet export reste utile pour réutiliser les tables ailleurs, mais je ne m'en sers plus pour la carte d'identité (voir partie 3).

In [ ]:
dump = DumpToCsv()  # dialecte postgres
tables = dump.extractTables("Datas/machine.sql")
{table: df.shape for table, df in tables.items()}

# Export CSV si besoin : dump.convert("Datas/machine.sql")

## 3. Carte d'identité des tables issues du SQL

### Pourquoi ne plus passer par le CSV ?

Ma première approche était : dump → CSV → `generateIdentities(csv)`. Mais la carte décrivait alors le **CSV intermédiaire**, pas le dump :
- **Source** pointait vers `Datas/dump/machine_20261120_….csv` au lieu de `Datas/machine.sql` ;
- **Filesize** et **Charset** étaient ceux du CSV que j'avais moi-même écrit, pas ceux du fichier reçu ;
- le nom de la carte contenait **deux horodatages** (`machine_20261120_152444_152444.md`) ;
- un aller-retour par le CSV transforme tout en texte, puis pandas doit re-deviner les types. Un `NULL` et une chaîne vide deviennent indiscernables, et un code `'007'` peut redevenir le nombre `7`.

### La nouvelle démarche

Grâce au découpage du générateur :
1. `collectFromDataFrame(df)` calcule tout ce qui vient des **données** de la table ;
2. les **setters** renseignent ce qui vient du **fichier d'origine** : `source` = le `.sql`, sa taille, son encodage, et un `name` du type `<dump>_<table>` ;
3. `write()` construit et écrit la carte.

In [ ]:
sql = Path("Datas/machine.sql")

for table, df in tables.items():
    generator.reset().collectFromDataFrame(df)
    generator.name = f"{sql.stem}_{table}"
    generator.source = sql
    generator.filesize = sql.stat().st_size
    generator.charset = IdentityGenerator._detect_charset(sql)
    print(generator.write())

## Limites et pistes d'amélioration

- **Détection de l'encodage** : seuls UTF-8 et ISO-8859-1 sont testés. Pour d'autres encodages, une librairie comme `charset-normalizer` serait plus fiable.
- **Inférence des types** : la règle `_id` et le test de date sur 100 valeurs sont des heuristiques simples. Une colonne comme `operator_badge` reste en TEXT alors que c'est aussi un identifiant.
- **Séparateur** : il est fixé pour toute l'instance ; on pourrait le détecter automatiquement avec `csv.Sniffer`.
- **Mémoire** : le CSV est chargé entièrement. Pour de très gros fichiers, il faudrait le lire par blocs (`chunksize`).
- **SQL** : les types déclarés dans les `CREATE TABLE` ne sont pas encore exploités. Ils pourraient remplacer l'inférence via le setter `columns`. Les colonnes absentes des `INSERT` (ex. `is_active`, `created_at` de `machine`, qui prennent leur valeur par défaut) n'apparaissent pas non plus dans la carte.
- **Filesize d'une table SQL** : c'est la taille du dump entier, partagé par toutes ses tables, et non celle de la table seule.
- **Modèle Markdown** : le gabarit est écrit en Python. Un moteur de templates (Jinja2) permettrait de le modifier sans toucher au code.
